In [ ]:
# ======================================================
# Notebook: Hyperparameter optimisation (CNN surrogate)
# Inputs: (30,4) | Output: (30,)
# Goal: minimise difference from baseline (maximize negative)
# ======================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (30,4)
y_raw = np.load("/mnt/data/initial_outputs.npy") # (30,)

# Transform objective
y = -y_raw
y = (y - y.mean()) / (y.std() + 1e-8)

# Reshape for CNN (batch, channel=1, height=4, width=1)
X_cnn = X.reshape(-1,1,4,1)

X_t = torch.tensor(X_cnn, dtype=torch.float32)
y_t = torch.tensor(y.reshape(-1,1), dtype=torch.float32)

# CNN surrogate
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(4,1)),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.fc = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = CNN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Train
model.train()
for _ in range(800):
    optimizer.zero_grad()
    pred = model(X_t)
    loss = criterion(pred, y_t)
    loss.backward()
    optimizer.step()

# Candidate generation
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(4)]
num_candidates = 5000
X_grid = np.column_stack([
    np.random.uniform(b[0], b[1], num_candidates) for b in bounds
])

X_grid_cnn = X_grid.reshape(-1,1,4,1)
X_grid_t = torch.tensor(X_grid_cnn, dtype=torch.float32)

# MC Dropout uncertainty
model.train()
samples = []

with torch.no_grad():
    for _ in range(30):
        samples.append(model(X_grid_t).numpy())

samples = np.stack(samples)
mean_pred = samples.mean(axis=0).flatten()
uncertainty = samples.std(axis=0).flatten()

# Acquisition
acquisition = mean_pred + 0.5 * uncertainty

# Select next (10,4)
top_idx = np.argsort(acquisition)[-10:]
next_points = X_grid[top_idx]

print("Next (10,4) hyperparameter candidates:")
print(next_points)